# Clean data validation

Checks the clean parquet files with DQX and writes the results to the same tables as `qbe_validation`.

1. Reads the settings (`config/clean_validation.yml`) and the rules (`config/clean_tables`, or the rules table).
2. Reads each table's files, e.g. `<clean_root>/<file_date>/<table>/part-*`.
3. Runs the rules and writes `dq_rule_summary`, `dq_threshold_results` and `dq_failed_records`.
4. Fails at the end if a table couldn't be read or a rule's column is missing (the reason is in the `error` column).

**Parameters:** `clean_root` (`file:/Workspace/...` or `/Volumes/...`), `file_date` (`yyyymmdd`, or `*`),
`results_catalog`, `rules_source` (`yaml` or `table`).

In [ ]:
%pip install databricks-labs-dqx==0.16.0

In [ ]:
%restart_python

In [ ]:
import json


def current_job_run_id() -> str:
    """The job run id when this notebook runs as a job task; "manual" for interactive runs."""
    try:
        context = json.loads(dbutils.notebook.entry_point.getDbutils().notebook().getContext().safeToJson())
        attributes = context.get("attributes", {})
        return str(attributes.get("multitaskParentRunId") or attributes.get("jobRunId") or "manual")
    except Exception:
        return "manual"


job_run_id = current_job_run_id()

In [ ]:
import re
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

import yaml
from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient
from pyspark.sql import functions as F

# Config paths are relative to use_cases/qbe, the parent of this notebook's folder (notebooks); its src folder
# holds the com.qbe.de package
SRC_DIR = Path.cwd().parent
sys.path.insert(0, str(SRC_DIR / "src"))
from com.qbe.de.rules import from_rule_rows, read_yaml_tables


def read_yaml(relative_path: str) -> dict:
    with open(SRC_DIR / relative_path) as yaml_file:
        return yaml.safe_load(yaml_file)


def describe(source: dict) -> str:
    """A source, for messages: the table <name>, or the <format> files at <path>."""
    return f"the table {source['table']}" if source.get("table") else f"the {source['format']} files at {source['path']}"


def rules_of(table_config: dict) -> list:
    """A table file's enabled rules, each with its column, business term and dataset name filled in."""
    rules = []
    for column, column_config in (table_config.get("columns") or {}).items():
        for rule in column_config.get("rules") or []:
            rules.append({"business_term": column_config.get("business_term"),
                          **rule, "column": column, "table": table_config["table"]})
    for rule in table_config.get("table_rules") or []:
        rules.append({**rule, "column": None, "table": table_config["table"]})
    return [rule for rule in rules if rule.get("enabled", True)]


def parameter(name: str, default: str) -> str:
    """A notebook parameter (widget) with a default; a job sets it with a job parameter of the same name."""
    try:
        dbutils.widgets.text(name, default)
        return dbutils.widgets.get(name)
    except NameError:               # no dbutils outside Databricks: use the default
        return default


def resolve(value, variables: dict):
    """Replaces ${name} in every string of a config with the variable's value; unknown names fail."""
    if isinstance(value, str):
        def lookup(match):
            if match.group(1) not in variables:
                raise ValueError(f"${{{match.group(1)}}} is not defined under variables in clean_validation.yml")
            return variables[match.group(1)]
        return re.sub(r"\$\{([^}]+)\}", lookup, value)
    if isinstance(value, dict):
        return {key: resolve(item, variables) for key, item in value.items()}
    if isinstance(value, list):
        return [resolve(item, variables) for item in value]
    return value


raw_settings = read_yaml("config/clean_validation.yml")["validation"]
variables = {name: parameter(name, str(default)) for name, default in (raw_settings.get("variables") or {}).items()}
settings = resolve(raw_settings, variables)
# The rules, from the YAML files in config/clean_tables or from the rules table's rows of this rule set
# (rules.source in clean_validation.yml). ${name} in either is resolved here.
RULE_SOURCES = ("yaml", "table")
rules_source, rule_set = settings["rules"]["source"], settings["rules"]["rule_set"]
if rules_source == "yaml":
    raw_table_configs = read_yaml_tables(SRC_DIR / settings["rules"]["tables_dir"])
elif rules_source == "table":
    rules_df = spark.read.table(settings["rules"]["table"])
    if "rule_set" not in rules_df.columns:
        raise ValueError(f"{settings['rules']['table']} has no rule_set column: run the qbe_clean_sync_rules job first")
    rule_rows = [row.asDict(recursive=True) for row in rules_df.where(F.col("rule_set") == rule_set).collect()]
    if not rule_rows:
        raise ValueError(f"No rules of rule_set {rule_set} in {settings['rules']['table']}: run the qbe_clean_sync_rules job first")
    raw_table_configs = from_rule_rows(rule_rows)
else:
    raise ValueError(f"rules.source must be one of {RULE_SOURCES}, got: {rules_source}")
table_configs = [resolve(table_config, variables) for table_config in raw_table_configs]
rules_by_table = {table_config["table"]: rules_of(table_config) for table_config in table_configs}


def source_name(source: dict) -> str:
    """A tables entry's name, checking it has one source: table, or files (name, path and format)."""
    if bool(source.get("table")) == bool(source.get("path")):
        raise ValueError(f"Each entry under tables in clean_validation.yml needs table or path (not both): {source}")
    if source.get("path") and not (source.get("name") and source.get("format")):
        raise ValueError(f"An entry with path under tables in clean_validation.yml needs name and format: {source}")
    return source.get("table") or source["name"]


# Each table's source, under tables in clean_validation.yml (as in a DQX config): a table, or files by name. Its
# record_columns are the source's, as the rules don't say where the data is read from
sources = {}
for source in settings.get("tables") or []:
    name = source_name(source)
    if name in sources:
        raise ValueError(f"{name} is under tables in clean_validation.yml more than once")
    sources[name] = source
for table_config in table_configs:
    table_config["source"] = sources.get(table_config["table"])
    table_config["record_columns"] = (table_config["source"] or {}).get("record_columns")

rule_ids = [rule["rule_id"] for table_rules in rules_by_table.values() for rule in table_rules]
duplicates = sorted({rule_id for rule_id in rule_ids if rule_ids.count(rule_id) > 1})
if duplicates:
    raise ValueError(f"rule_id used more than once in the rules: {', '.join(duplicates)}")

RESULT_MODES = ("append", "overwrite")
for result_name, result in settings["results"].items():
    if result.get("mode", "append") not in RESULT_MODES:
        raise ValueError(f"results.{result_name}.mode must be one of {RESULT_MODES}, got: {result.get('mode')}")

dq_engine = DQEngine(WorkspaceClient())
run_id = str(uuid.uuid4())
run_ts = datetime.now(timezone.utc).replace(tzinfo=None)

print(f"run_id={run_id} job_run_id={job_run_id} variables={variables} rules from {rules_source} ({rule_set}): "
      f"tables={len(table_configs)} rules={len(rule_ids)}")
for table_config in table_configs:
    print(f"{table_config['table']}: " + (describe(table_config["source"]) if table_config["source"] else "no source under tables"))
for name in sorted(set(sources) - set(rules_by_table)):
    print(f"{name}: no rules, not checked")
for result_name, result in settings["results"].items():
    print(f"{result_name}: " + (f"{result['table']} ({result.get('mode', 'append')})" if result.get("enabled", True) else "not persisted"))

In [ ]:
# Schemas of the results tables: qbe_validation's, plus source, cde_name, domain and LOB. rule_summary (one row per
# rule per run: what the run measured), threshold_results (one row per rule per run: the pass rate against the rule's
# thresholds) and failed_records (one row per failing record per rule)
SOURCE, DOMAIN, LOB = "GWCC", "Claims", "WC"     # cde_name is the rule's column
CLEAN_COLUMNS = ", source string, cde_name string, domain string, LOB string"
SUMMARY_SCHEMA = (
    "dq_run_id string, dq_run_ts timestamp, dq_job_run_id string, rule_id string, rule_name string, "
    "business_term string, dq_dimension string, source_table string, column_name string, "
    "criticality string, rows_checked long, rows_in_scope long, rows_failed long, pass_rate double, error string"
    + CLEAN_COLUMNS
)
THRESHOLD_RESULTS_SCHEMA = (
    "dq_run_id string, dq_run_ts timestamp, dq_job_run_id string, rule_id string, rule_name string, "
    "source_table string, column_name string, criticality string, pass_rate double, "
    "minimum_threshold double, target_threshold double, threshold_status string"
    + CLEAN_COLUMNS
)
FAILED_RECORDS_SCHEMA = (
    "dq_run_id string, dq_run_ts timestamp, dq_job_run_id string, source_table string, rule_id string, "
    "rule_name string, criticality string, message string, columns array<string>, record string"
    + CLEAN_COLUMNS
)
MAX_COLUMNS_LISTED = 50             # of the columns found, in a missing column's error


class DatasetError(Exception):
    """A table none of whose rules can run (no source, its source can't be read, or record_columns aren't in it):
    the message is each rule's error."""


def reason(error: Exception) -> str:
    """An exception's message for the error column: its first line (Spark's starts with the error class, e.g.
    [PATH_NOT_FOUND]), without the Java stack trace."""
    lines = [line.strip() for line in str(error).splitlines() if line.strip()]
    return (lines[0] if lines else type(error).__name__)[:1000]


def read_dataset(table_config: dict):
    """The table's source as a DataFrame: the table, or the files (and how many were read, else None); DatasetError
    when there's no source or it can't be read."""
    source = table_config["source"]
    if not source:
        raise DatasetError(f"No source for {table_config['table']} under tables in clean_validation.yml")
    try:
        if source.get("table"):
            df, files = spark.read.table(source["table"]), None
            df.columns              # reads the table's schema: a missing table fails here, not in the checks
        else:
            options = {key: str(value) for key, value in (source.get("options") or {}).items()}
            df = spark.read.format(source["format"]).options(**options).load(source["path"])
            files = len(df.inputFiles())
    except Exception as error:      # e.g. [PATH_NOT_FOUND], [UNABLE_TO_INFER_SCHEMA], [TABLE_OR_VIEW_NOT_FOUND]
        hint = ""
        if "UNABLE_TO_INFER_SCHEMA" in str(error):
            hint = (f". The folder is empty, or its files are in subfolders that aren't partitions (key=value), e.g. a "
                    f"folder per month or day: read them with a glob, {source['path'].rstrip('/')}/*/, or options "
                    f'{{recursiveFileLookup: "true"}}')
        raise DatasetError(f"Couldn't read {describe(source)}: {reason(error)}{hint}") from error
    if files == 0:
        raise DatasetError(f"No files in {describe(source)}")
    return df, files


def columns_of(rule: dict) -> list:
    """The columns a rule names: its column, check.for_each_column, and a column/columns argument. A column only
    used inside a sql_expression isn't known here: if it's missing, the check's error says so."""
    arguments = rule["check"].get("arguments") or {}
    names = [rule["column"]] if rule["column"] else []
    names += rule["check"].get("for_each_column") or []
    names += [arguments["column"]] if isinstance(arguments.get("column"), str) else []
    names += [name for name in arguments.get("columns") or [] if isinstance(name, str)]
    return list(dict.fromkeys(names))


def not_in(names: list, df) -> list:
    """The names that aren't columns of df (compared ignoring case, as Spark does)."""
    present = {column.lower() for column in df.columns}
    return [name for name in names if name.lower() not in present]


def columns_found(df) -> str:
    listed = ", ".join(df.columns[:MAX_COLUMNS_LISTED])
    return listed + (f", ... ({len(df.columns)} in all)" if len(df.columns) > MAX_COLUMNS_LISTED else "")


def to_dqx_check(rule: dict) -> dict:
    """The DQX check for a rule: named by rule_id, on its column (column rules) or on each column in
    check.for_each_column (table rules), with the rule's DQX filter."""
    function = rule["check"]["function"]
    arguments = dict(rule["check"].get("arguments") or {})
    for_each_column = rule["check"].get("for_each_column")
    if for_each_column and rule["column"]:
        raise ValueError(f"{rule['rule_id']}: for_each_column is for table_rules; a column file's rules are on its column")
    if rule["column"] and function == "sql_expression":
        arguments.setdefault("columns", [rule["column"]])
    elif rule["column"]:
        arguments.setdefault("column", rule["column"])
    check = {"name": rule["rule_id"], "criticality": rule.get("criticality", "error"),
             "check": {"function": function, "arguments": arguments}}
    # DQX runs the check once per column, every copy named rule_id: a record fails the rule if any column fails
    if for_each_column:
        check["check"]["for_each_column"] = list(for_each_column)
    if rule.get("filter"):
        check["filter"] = rule["filter"]
    return check


def threshold_status(pass_rate, rule: dict) -> str:
    if pass_rate is None:
        return "no_rows"
    minimum, target = rule.get("minimum_threshold"), rule.get("target_threshold")
    if minimum is not None and pass_rate < minimum:
        return "below_minimum"
    if target is not None and pass_rate < target:
        return "below_target"
    return "met"


def column_name(rule: dict) -> str:
    return rule["column"] or ", ".join(rule["check"].get("for_each_column")
                                       or rule["check"].get("arguments", {}).get("columns") or [])


def summary_row(rule: dict, table_config: dict, **values) -> dict:
    as_float = lambda value: float(value) if value is not None else None
    return {
        "dq_run_id": run_id, "dq_run_ts": run_ts, "dq_job_run_id": job_run_id,
        "rule_id": rule["rule_id"], "rule_name": rule.get("rule_name"), "business_term": rule.get("business_term"),
        "dq_dimension": rule.get("dq_dimension"), "source_table": table_config["table"],
        "column_name": column_name(rule),
        "criticality": rule.get("criticality", "error"),
        "minimum_threshold": as_float(rule.get("minimum_threshold")), "target_threshold": as_float(rule.get("target_threshold")),
        "rows_checked": None, "rows_in_scope": None, "rows_failed": None, "pass_rate": None,
        "threshold_status": None, "error": None,
        "source": SOURCE, "cde_name": column_name(rule), "domain": DOMAIN, "LOB": LOB, **values,
    }


def error_rows(table_rules: list, table_config: dict, error: str) -> list:
    return [summary_row(rule, table_config, threshold_status="error", error=error[:2000]) for rule in table_rules]


def rows_to_df(rows: list, schema: str):
    """A DataFrame of the schema's columns from rows (dicts) that may hold more keys."""
    names = [field.name for field in spark.createDataFrame([], schema).schema.fields]
    return spark.createDataFrame([{name: row[name] for name in names} for row in rows], schema)


def is_rule(rule_id: str):
    """A one-argument predicate for F.exists: whether a DQX result belongs to the rule."""
    return lambda result: result["name"] == rule_id


def validate_dataset(table_config: dict, table_rules: list) -> tuple:
    """Reads a table's source and runs its rules in one DQX pass: one summary row per rule, and the failing records
    (None when no rule ran). A rule whose columns aren't in the source is an error row; the others still run.
    DatasetError when no rule can run."""
    df, files = read_dataset(table_config)
    source = describe(table_config["source"])
    print(f"{table_config['table']}: {len(df.columns)} columns" + (f", {files} files" if files else "") + f" in {source}")
    missing_record_columns = not_in(table_config.get("record_columns") or [], df)
    if missing_record_columns:
        raise DatasetError(f"record_columns not in {source}: {', '.join(missing_record_columns)} "
                           f"(columns found: {columns_found(df)})")

    rows, runnable = [], []
    for rule in table_rules:
        missing = not_in(columns_of(rule), df)
        if missing:
            rows += error_rows([rule], table_config, f"Column(s) not in {source}: {', '.join(missing)} "
                                                     f"(columns found: {columns_found(df)})")
        else:
            runnable.append(rule)
    if not runnable:
        return rows, None
    try:
        checked_rows, failures = check_rules(df, table_config, runnable)
    except Exception as error:      # e.g. a corrupt file, or a sql_expression naming a column the source doesn't have
        return rows + error_rows(runnable, table_config, f"Couldn't check {source}: {reason(error)}"), None
    return rows + checked_rows, failures


def check_rules(df, table_config: dict, table_rules: list) -> tuple:
    """Runs rules on df in one DQX pass; returns one summary row per rule and the failing records."""
    checks = [to_dqx_check(rule) for rule in table_rules]
    status = DQEngine.validate_checks(checks)
    if status.has_errors:
        raise ValueError(f"Invalid DQX checks: {status}")
    checked = dq_engine.apply_checks_by_metadata(df, checks)
    if isinstance(checked, tuple):      # DQX adds an Observation when an observer is configured
        checked = checked[0]

    # One pass over the data: rows checked, and per rule the rows in scope and the rows that failed it
    aggregations = [F.count("*").alias("rows_checked")]
    for index, rule in enumerate(table_rules):
        in_scope = F.expr(rule["filter"]) if rule.get("filter") else F.lit(True)
        failed = F.exists("_errors", is_rule(rule["rule_id"])) | F.exists("_warnings", is_rule(rule["rule_id"]))
        aggregations += [F.count(F.when(in_scope, 1)).alias(f"in_scope_{index}"), F.count(F.when(failed, 1)).alias(f"failed_{index}")]
    counts = checked.agg(*aggregations).first()

    rows = []
    for index, rule in enumerate(table_rules):
        in_scope, failed = counts[f"in_scope_{index}"], counts[f"failed_{index}"]
        pass_rate = round((in_scope - failed) / in_scope * 100, 4) if in_scope else None
        rows.append(summary_row(rule, table_config, rows_checked=counts["rows_checked"], rows_in_scope=in_scope,
                                rows_failed=failed, pass_rate=pass_rate, threshold_status=threshold_status(pass_rate, rule)))
    return rows, failures_of(checked, table_config, table_rules)


def failures_of(checked, table_config: dict, table_rules: list):
    """One row per failing record per rule (FAILED_RECORDS_SCHEMA): the rule, its message, and the record as JSON."""
    source_columns = [c for c in checked.columns if c not in ("_errors", "_warnings")]
    record_columns = table_config.get("record_columns") or source_columns
    record = F.to_json(F.struct(*[F.col(f"`{c}`") for c in record_columns]))
    rule_names = F.create_map(*[F.lit(v) for rule in table_rules for v in (rule["rule_id"], rule.get("rule_name") or "")])
    cde_names = F.create_map(*[F.lit(v) for rule in table_rules for v in (rule["rule_id"], column_name(rule))])
    results = (
        checked.select(record.alias("record"), F.explode("_errors").alias("result"), F.lit("error").alias("criticality"))
        .unionByName(checked.select(record.alias("record"), F.explode("_warnings").alias("result"), F.lit("warn").alias("criticality")))
    )
    failed_records = results.select(
        F.lit(run_id).alias("dq_run_id"),
        F.lit(run_ts).alias("dq_run_ts"),
        F.lit(job_run_id).alias("dq_job_run_id"),
        F.lit(table_config["table"]).alias("source_table"),
        F.col("result.name").alias("rule_id"),
        F.element_at(rule_names, F.col("result.name")).alias("rule_name"),
        "criticality",
        F.col("result.message").alias("message"),
        F.col("result.columns").alias("columns"),
        "record",
        F.lit(SOURCE).alias("source"),
        F.element_at(cde_names, F.col("result.name")).alias("cde_name"),
        F.lit(DOMAIN).alias("domain"),
        F.lit(LOB).alias("LOB"),
    )
    # Cast to FAILED_RECORDS_SCHEMA so the table's columns and types don't depend on DQX's result types
    schema = spark.createDataFrame([], FAILED_RECORDS_SCHEMA).schema
    return failed_records.select(*[F.col(field.name).cast(field.dataType).alias(field.name) for field in schema.fields])


def persist(result_name: str, df) -> None:
    """Writes one result (rule_summary, threshold_results or failed_records) as configured under results.<result_name>."""
    result = settings["results"][result_name]
    if not result.get("enabled", True):
        return
    mode = result.get("mode", "append")
    writer = df.write.mode(mode)
    writer = writer.option("overwriteSchema", "true") if mode == "overwrite" else writer.option("mergeSchema", "true")
    writer.saveAsTable(result["table"])

In [ ]:
summary_rows, failures_dfs = [], []
for table_config in table_configs:
    table_rules = rules_by_table[table_config["table"]]
    if not table_rules:
        continue
    try:
        rows, failures = validate_dataset(table_config, table_rules)
        summary_rows += rows
        if failures is not None:
            failures_dfs.append(failures)
    except DatasetError as error:   # no source, it can't be read, or record_columns aren't in it
        summary_rows += error_rows(table_rules, table_config, str(error))
    except Exception as error:
        summary_rows += error_rows(table_rules, table_config, f"Couldn't check {table_config['table']}: {reason(error)}")

# Each result is written once per run, so "overwrite" keeps the whole run, not just the last table
if failures_dfs:
    all_failures = failures_dfs[0]
    for failures in failures_dfs[1:]:
        all_failures = all_failures.unionByName(failures)
    persist("failed_records", all_failures)

# summary_rows hold each rule's counts and threshold result; each table gets its own columns
summary = rows_to_df(summary_rows, SUMMARY_SCHEMA)
persist("rule_summary", summary)
threshold_results = rows_to_df(summary_rows, THRESHOLD_RESULTS_SCHEMA)
persist("threshold_results", threshold_results)
display(summary)
display(threshold_results)

In [ ]:
# The rules that couldn't run, with why: the notebook (and its job) fails, after the results are written
errors = [row for row in summary_rows if row["threshold_status"] == "error"]
if errors:
    display(rows_to_df(errors, SUMMARY_SCHEMA).select("source_table", "rule_id", "column_name", "error"))
    details = "\n".join(f"  {row['source_table']} {row['rule_id']}: {row['error']}" for row in errors)
    raise RuntimeError(f"{len(errors)} rule(s) could not run (the error column of the rule summary has the same):\n{details}")

breached = [row["rule_id"] for row in summary_rows
            if row["threshold_status"] in ("below_target", "below_minimum") and row["criticality"] == "error"]
if breached and settings.get("fail_on_threshold_breach", False):
    raise RuntimeError(f"Rules below their target threshold: {', '.join(breached)}")